In [ ]:
import geopandas as gpd
import geohash #--> !pip install python-geohash 
import gzip
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import seaborn as sns
from shapely.geometry import Point, Polygon
from shapely import wkt
from geopy.geocoders import Nominatim
from uszipcode import SearchEngine
import plotly.express as px
from sodapy import Socrata # crime data API
import requests
from io import StringIO
import gdown


In [ ]:
########################### AIRBNB DATA ##########################################################################################################
# source 1+2+3: http://insideairbnb.com/get-the-data/
airbnb_data_path = "./data/airbnb_data/"
listings_csv_path = airbnb_data_path + "listings.csv.gz"
############################ CRIME DATA ##########################################################################################################
crime_data_path = "./data/crime_data/"
# source 1: https://data.cityofchicago.org/Public-Safety/Crimes-2001-to-Present/ijzp-q8t2
response = requests.get('https://data.cityofchicago.org/resource/ijzp-q8t2.csv')
assert response.status_code == 200
data = response.content.decode('utf-8')
df = pd.read_csv(StringIO(data))
# df.to_csv(crime_data_path + "Crimes_-_2001_to_Present.csv") # uncomment to store data
#crime_csv_path = crime_data_path + "Crimes_-_2001_to_Present.csv"
# source 2: https://data.cityofchicago.org/Public-Safety/Chicago-Police-Department-Illinois-Uniform-Crime-R/c7ck-438e/
# below: retrieve first 1000 rows
file_id = '1Nl7eEWJFA8709eAy9EbdAXi7ZORRul0n'
output_file = 'crime.csv'

# Download the file from Google Drive
gdown.download(f'https://drive.google.com/uc?id={file_id}', output_file, quiet=False)

response = requests.get('https://data.cityofchicago.org/resource/c7ck-438e.csv')
assert response.status_code == 200
data = response.content.decode('utf-8')
df = pd.read_csv(StringIO(data))
df.to_csv(crime_data_path + "Chicago_Police_Department_-_Illinois_Uniform_Crime_Reporting__IUCR__Codes.csv") 
crime_codes_csv_path = crime_data_path + "Chicago_Police_Department_-_Illinois_Uniform_Crime_Reporting__IUCR__Codes.csv"

########################### POPULATION DATA ########################################################################################################
population_data_path = "./data/population_data/"
# source 1: https://data.cityofchicago.org/Facilities-Geographic-Boundaries/Population-by-2010-Census-Block/5yjb-v3mj/about_data 
population_census_csv_path = population_data_path + "Population_by_2010_Census_Block.csv"
# source 2: https://data.cityofchicago.org/Facilities-Geographic-Boundaries/Boundaries-Census-Tracts-2010/5jrd-6zik 
census_boundaries_csv_path = population_data_path + "CensusTractsTIGER2010.csv"

In [ ]:
import pandas as pd
import json
import os

VERBOSE = True
CRIME_PATH = 'crime.csv'
LISTINGS_PATH = 'data/airbnb_data/listings.csv.gz'
OUTPUT_PATH = 'src/data/chicago_timeseries.json'
    

def vprint(*args, **kwargs):
    if VERBOSE:
        vprint(*args, **kwargs)

def generate_data():

    vprint("1 Loading data")
    os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

    vprint("\t1.1 Loading AirBnB data")
    try:
        df_listings = pd.read_csv(LISTINGS_PATH, compression='gzip')
    except Exception as _:
        df_listings = pd.read_csv('data/airbnb_data/listings.csv')

    vprint("\t1.2 Loading crime data")
    crime_cols = ['Date', 'Primary Type', 'Latitude', 'Longitude']
    df_crime = pd.read_csv(CRIME_PATH, usecols=crime_cols) # subset

    df_crime = df_crime.rename(columns={
        'Latitude': 'latitude', 
        'Longitude': 'longitude',
        'Primary Type': 'Primary Type'
    })
    df_crime = df_crime.dropna(subset=[
        'latitude', 'longitude'
    ])

    vprint("3 Filtering data")
    SELECTED_CRIMES = ['HOMICIDE', 'BATTERY', 'ASSAULT', 'ROBBERY', 'BURGLARY']
    df_crime_filtered = df_crime[df_crime['Primary Type'].isin(SELECTED_CRIMES)].copy()
    df_crime_filtered['Date'] = pd.to_datetime(df_crime_filtered['Date'])   

    # quarterly periods (aggregated)
    df_crime_filtered['period'] = df_crime_filtered['Date'].dt.to_period('Q').astype(str)
    periods = sorted(df_crime_filtered['period'].unique())

    vprint("4 Structuring data")
    
    airbnb_data = {
        "count": len(df_listings),
        "locations": df_listings[['latitude', 'longitude', 'price', 'room_type', 'neighbourhood']].rename(
            columns={'latitude': 'lat', 'longitude': 'lon', 'neighbourhood': 'neighborhood'}
        ).fillna(0).to_dict('records')
    }

    output_data = {
        "periods": periods,
        "crime_types": SELECTED_CRIMES,
        "airbnb": airbnb_data,
        "crimes": []
    }

    MAX_CRIMES_PER_PERIOD = 3_000 # Limit points for browser performance
    for period in periods:
        period_data = df_crime_filtered[df_crime_filtered['period'] == period]
        # sample if too many
        if len(period_data) > MAX_CRIMES_PER_PERIOD:
            period_data = period_data.sample(n=MAX_CRIMES_PER_PERIOD, random_state=999)
        
        output_data["crimes"].append({
            "period": period,
            "count": int(len(period_data)),
            "locations": period_data[['latitude', 'longitude', 'Primary Type']].rename(
                columns={'latitude': 'lat', 'longitude': 'lon', 'Primary Type': 'type'}
            ).to_dict('records')
        })

    vprint(f"5 Exporting {len(periods)} periods to {OUTPUT_PATH}")
    with open(OUTPUT_PATH, 'w') as f:
        json.dump(output_data, f)

    vprint("6 Done - LOL")


generate_data()